In [6]:
import pandas as pd 
import numpy as np
import sklearn
import imblearn 

main = pd.read_csv(r'M:\ML_scripts\mordred_mold2_rdkit_descr_tob_5889.csv').drop(columns=['Unnamed: 0'])
df = main.copy()

# Downsample

In [8]:
# downsampling
from imblearn.under_sampling import RepeatedEditedNearestNeighbours as RENN, NearMiss

# renn = RENN(sampling_strategy='majority', n_neighbors=29, max_iter=5000, kind_sel='all', n_jobs=-1)

nearMiss = NearMiss(sampling_strategy='majority', version=2, n_neighbors=5, n_jobs=-1)

df_x = df.drop(columns=['Structure'])
df_y = df['Class']

df_str = pd.DataFrame(df['Structure'])

df_x_res, df_y_res = nearMiss.fit_resample(df_x, df_y)

index = nearMiss.sample_indices_
df_smiles = df_str.iloc[index]
df_smiles = df_smiles.reset_index(drop=True)

df_res = pd.concat([df_smiles, df_x_res], axis=1)

In [23]:
# for the excluded samples

excluded = df.drop(index = index)
excluded.to_csv(r'M:\ML_scripts\MODEL DATA\sample_then_split\excluded_samples.csv')

In [10]:
from sklearn.model_selection import train_test_split

res_train, res_test = train_test_split(df_res, test_size=0.2, stratify=df_res['Class'], random_state=42)

# Remaining processing steps

In [ ]:
# train set processing

df_str = res_train['Structure'] # smiles
df_y = res_train['Class'] # classif 

from sklearn.feature_selection import VarianceThreshold

def variance(data, threshold=(0.1)):

    sel = VarianceThreshold(threshold)  # removing features of low variance
    sel.fit_transform(data)
    
    return data[data.columns[sel.get_support(indices=True)]]

def findCorrelation(corr, cutoff=0.9, exact=None):
    
    def _findCorrelation_fast(corr, avg, cutoff):

        combsAboveCutoff = corr.where(lambda x: (np.tril(x)==0) & (x > cutoff)).stack().index

        rowsToCheck = combsAboveCutoff.get_level_values(0)
        colsToCheck = combsAboveCutoff.get_level_values(1)

        msk = avg[colsToCheck] > avg[rowsToCheck].values
        deletecol = pd.unique(np.r_[colsToCheck[msk], rowsToCheck[~msk]]).tolist()

        return deletecol


    def _findCorrelation_exact(corr, avg, cutoff):

        x = corr.loc[(*[avg.sort_values(ascending=False).index]*2,)]

        if (x.dtypes.values[:, None] == ['int64', 'int32', 'int16', 'int8']).any():
            x = x.astype(float)

        x.values[(*[np.arange(len(x))]*2,)] = np.nan

        deletecol = []
        for ix, i in enumerate(x.columns[:-1]):
            for j in x.columns[ix+1:]:
                if x.loc[i, j] > cutoff:
                    if x[i].mean() > x[j].mean():
                        deletecol.append(i)
                        x.loc[i] = x[i] = np.nan
                    else:
                        deletecol.append(j)
                        x.loc[j] = x[j] = np.nan
        return deletecol

    
    if not np.allclose(corr, corr.T) or any(corr.columns!=corr.index):
        raise ValueError("correlation matrix is not symmetric.")
        
    acorr = corr.abs()
    avg = acorr.mean()
        
    if exact or exact is None and corr.shape[1]<100:
        return _findCorrelation_exact(acorr, avg, cutoff)
    else:
        return _findCorrelation_fast(acorr, avg, cutoff)

# variance
train = res_train.drop(columns=['Structure','Class'])
train_var = variance(train)

# correlation
train_corr = train_var.corr()

hc = findCorrelation(train_corr, cutoff=0.9, exact=True)
train_var_corr = train_var.drop(columns=hc)

# scaling
from sklearn.preprocessing import RobustScaler

RobScaler = RobustScaler(unit_variance=True)
RobScaler.fit(train_var_corr)

train_var_corr_scal = RobScaler.transform(train_var_corr)
train_var_corr_scal = pd.DataFrame(train_var_corr_scal, columns=train_var_corr.columns)

str_clf = pd.concat([df_str, df_y], axis=1)
str_clf = str_clf.reset_index(drop=True)

df2 = pd.concat([str_clf, train_var_corr_scal], axis=1)

# test set processing

df_test = res_test[df2.columns]

test_smiles = df_test['Structure']
test_y = df_test['Class']

df_test_scal = RobScaler.transform(df_test.drop(columns=['Structure', 'Class']))
df_test_scal = pd.DataFrame(df_test_scal, columns=df_test.drop(columns=['Structure', 'Class']).columns)

test_smi_class = pd.concat([test_smiles, test_y], axis=1)
test_smi_class = test_smi_class.reset_index(drop=True)

test_set = pd.concat([test_smi_class, df_test_scal], axis=1)

In [ ]:
# df2.to_csv(r'M:\ML_scripts\MODEL DATA\sample_then_split\train_set.csv')
# test_set.to_csv(r'M:\ML_scripts\MODEL DATA\sample_then_split\test_set.csv')